In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
!pip install gradio pandas plotly

Mounted at /content/drive


In [3]:
!pip install gradio pandas plotly scikit-learn numpy pyarrow

💡 Gợi ý Product ID có luật:
e53e557d5a159f5aa2c5e995dfdf244b | 36f60d45225e60c7da4558b070ce4b60 | 35afc973633aaeb6b877ff57b2793310

In [ ]:
import gradio as gr
import pandas as pd
import plotly.express as px
import os
import shutil

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALSModel
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer
from pyspark.ml.fpm import FPGrowthModel
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType
from pyspark.ml import PipelineModel
from pyspark.ml.tuning import CrossValidatorModel
from functools import reduce
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
import time

# ========================== KHỞI TẠO SPARK ==========================
spark = SparkSession.builder.appName("Olist_Full_System_Nhom5").getOrCreate()

# ========================== ĐƯỜNG DẪN ==========================
PATH_MODELS = "/content/drive/MyDrive/BigData_Nhom5/models"
PATH_STREAMLIT = "/content/drive/MyDrive/BigData_Nhom5/models/streamlit_data_new"
DATA_PATH = "/content/drive/MyDrive/BigData_Nhom5/data/datasetCuoiKy/"
ALS_MODEL_PATH = os.path.join(PATH_MODELS, "als_model_new")
PATH_REG = os.path.join(PATH_MODELS, "best_regression_model")
PATH_CLS = os.path.join(PATH_MODELS, "best_model.classification")

# ========================== LOAD MODELS DỰ ĐOÁN (Cách cũ - ổn định) ==========================
def load_model_any(path):
    try:
        cv = CrossValidatorModel.load(path)
        return cv.bestModel
    except:
        try:
            return PipelineModel.load(path)
        except Exception as e:
            print(f"Không load được model tại {path}: {e}")
            return None

reg_model = load_model_any(PATH_REG)
best_cls_model = load_model_any(PATH_CLS)

# ========================== SCHEMA & HÀM DỰ ĐOÁN (ĐÃ SỬA LỖI KIỂU DỮ LIỆU) ==========================
full_schema = StructType([
    StructField("total_price", DoubleType(), True),
    StructField("total_order_value", DoubleType(), True),
    StructField("total_freight_value", DoubleType(), True),
    StructField("num_items", IntegerType(), True),
    StructField("product_weight_g", DoubleType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("seller_state", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("delivery_delay_days", DoubleType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("is_late_delivery", IntegerType(), True)
])

def predict_combined(mode, price, freight, items, weight, installments, pay_type, cat,
                     c_state, s_state, status, delay, photos, is_late):
    if reg_model is None or best_cls_model is None:
        return "❌ Không load được model dự đoán. Kiểm tra đường dẫn model trên Drive."
    try:
        input_dict = {
            "total_price": float(price),
            "total_order_value": float(price),
            "total_freight_value": float(freight),
            "num_items": int(items),
            "product_weight_g": float(weight),
            "payment_installments": int(installments),
            "payment_type": str(pay_type),
            "product_category_name_english": str(cat),
            "customer_state": str(c_state),
            "seller_state": str(s_state),
            "order_status": str(status),
            "delivery_delay_days": float(delay),
            "product_photos_qty": int(photos),
            "is_late_delivery": int(is_late)
        }

        data = [tuple(input_dict.values())]
        df = spark.createDataFrame(data, full_schema)

        if "Regression" in mode:
            result = reg_model.transform(df)
            pred_val = result.select("prediction").first()[0]
            return f"💰 GIÁ TRỊ THANH TOÁN DỰ KIẾN: {pred_val:.2f} R$\n(Dự đoán bởi Decision Tree)"
        else:
            result = best_cls_model.transform(df)
            pred_label = result.select("prediction").first()[0]
            status_text = "⭐ HÀI LÒNG (Review cao)" if pred_label >= 0.5 else "📉 KHÔNG HÀI LÒNG (Review thấp)"
            return f"🎭 TRẠNG THÁI KHÁCH HÀNG: {status_text}\n(Dự đoán bởi Logistic Regression)"
    except Exception as e:
        return f"⚠️ Lỗi dự đoán: {str(e)}"

# ========================== LOAD DASHBOARD DATA ==========================
df_dashboard = pd.read_parquet(os.path.join(PATH_STREAMLIT, "master_df.parquet"))
cols = df_dashboard.columns.tolist()
col_price = next((c for c in cols if 'price' in c.lower() or 'value' in c.lower()), None)
col_date = next((c for c in cols if 'timestamp' in c.lower() or 'date' in c.lower()), None)
col_review = next((c for c in ['review_score', 'rating'] if c in cols), None)
col_cat = next((c for c in ['product_category_name_english', 'product_category_name'] if c in cols), None)

def create_dashboard_plots():
    plots = []
    if col_price and col_date:
        df_temp = df_dashboard.copy()
        df_temp[col_date] = pd.to_datetime(df_temp[col_date])
        revenue_ts = df_temp.resample('M', on=col_date)[col_price].sum().reset_index()
        fig1 = px.line(revenue_ts, x=col_date, y=col_price, title='Tổng doanh thu theo tháng')
        plots.append(fig1)
    else:
        plots.append(px.scatter(title="Không tìm thấy cột Doanh thu/Ngày"))

    if col_review:
        fig2 = px.histogram(df_dashboard, x=col_review, title='Phân bố điểm đánh giá',
                            color_discrete_sequence=['#FFA15A'], nbins=5)
        plots.append(fig2)
    else:
        plots.append(px.scatter(title="Không tìm thấy cột Review"))

    if col_cat:
        top_cats = df_dashboard[col_cat].value_counts().nlargest(10).reset_index()
        top_cats.columns = [col_cat, 'count']
        fig3 = px.bar(top_cats, x=col_cat, y='count', title='Top 10 ngành hàng bán chạy',
                      color='count', color_continuous_scale='Viridis')
        plots.append(fig3)
    else:
        plots.append(px.scatter(title="Không tìm thấy cột Ngành hàng"))
    return plots[0] if len(plots)>0 else None, plots[1] if len(plots)>1 else None, plots[2] if len(plots)>2 else None

# ========================== CLUSTERING ==========================
def handle_clustering(file_obj, n_clusters):
    if file_obj is None:
        return None, "Vui lòng upload file CSV", None
    df = pd.read_csv(file_obj.name)
    required_cols = ['Recency', 'Frequency', 'Monetary']
    if not all(col in df.columns for col in required_cols):
        return None, f"File cần có các cột: {required_cols}", None

    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(df[required_cols])
    kmeans = KMeans(n_clusters=int(n_clusters), random_state=42, n_init=10)
    df['Cluster'] = kmeans.fit_predict(data_scaled).astype(str)

    fig_3d = px.scatter_3d(df, x='Recency', y='Frequency', z='Monetary',
                           color='Cluster', title=f'Kết quả phân cụm (K={n_clusters})',
                           opacity=0.8, color_discrete_sequence=px.colors.qualitative.Safe)

    summary = df.groupby('Cluster').agg({
        'Recency':'mean', 'Frequency':'mean', 'Monetary':'mean', 'Cluster':'count'
    }).rename(columns={'Cluster': 'Số lượng KH'}).reset_index()
    summary[['Recency', 'Frequency', 'Monetary']] = summary[['Recency', 'Frequency', 'Monetary']].round(2)

    return fig_3d, summary, df.head(10)

# ========================== ALS RECOMMENDATION ==========================
# ========================== ALS RECOMMENDATION (SỬA HOÀN CHỈNH - ĐỒNG BỘ VỚI TRAIN) ==========================
als = None
user_map = None
item_map = None
product_info = None

def reload_als_for_recommend():
    """Load model và mapping sau khi train"""
    global als, user_map, item_map, product_info

    try:
        als = ALSModel.load(ALS_MODEL_PATH)
        print("✓ Load model ALS mới từ train thành công")

        # Load master data (có customer_unique_id + product_id)
        master_spark = spark.read.parquet(os.path.join(PATH_STREAMLIT, "master_df.parquet"))

        u_indexer = StringIndexer(inputCol="customer_unique_id", outputCol="user_index", handleInvalid="keep")
        i_indexer = StringIndexer(inputCol="product_id", outputCol="item_index", handleInvalid="keep")

        master_indexed = u_indexer.fit(master_spark).transform(master_spark)
        master_indexed = i_indexer.fit(master_indexed).transform(master_indexed)

        user_map = master_indexed.select(
            F.trim(F.col("customer_unique_id")).alias("customer_unique_id"),
            "user_index"
        ).distinct()

        item_map = master_indexed.select("product_id", "item_index").distinct()

        # Thông tin sản phẩm
        cat_col = next((c for c in master_indexed.columns if 'category' in c.lower() and 'english' in c.lower()), 'product_id')
        price_col = next((c for c in ['total_price', 'price'] if c in master_indexed.columns), None)

        info_cols = ["product_id", cat_col]
        if price_col:
            info_cols.append(price_col)

        product_info = master_indexed.select(*info_cols).distinct().toPandas()
        if price_col:
            product_info = product_info.rename(columns={cat_col: 'Ngành hàng', price_col: 'Giá (R$)'})
        else:
            product_info = product_info.rename(columns={cat_col: 'Ngành hàng'})

        print(f"✓ Mapping sẵn sàng - Số user: {user_map.count():,} | Số item: {item_map.count():,}")
        return True

    except Exception as e:
        print(f"❌ Lỗi load mapping: {e}")
        return False


# Hàm gợi ý (đã cải tiến debug)
def recommend_products(customer_id):
    if als is None or user_map is None:
        return pd.DataFrame({"Thông báo": ["❌ Model ALS chưa được load. Vui lòng train lại ở tab Admin trước."]})

    if not customer_id or str(customer_id).strip() == "":
        return pd.DataFrame({"Thông báo": ["⚠️ Vui lòng nhập mã khách hàng"]})

    try:
        search_id = str(customer_id).strip()

        # Kiểm tra khách hàng có tồn tại không
        user_rows = user_map.filter(F.col("customer_unique_id") == search_id).select("user_index").collect()

        if not user_rows:
            return pd.DataFrame({"Thông báo": [f"❌ Không tìm thấy khách hàng '{search_id}' trong dữ liệu mapping"]})

        user_index = user_rows[0][0]

        user_df = spark.createDataFrame([(user_index,)], ["user_index"])
        recs = als.recommendForUserSubset(user_df, 10)

        if recs.count() == 0:
            return pd.DataFrame({"Thông báo": ["💡 Model không tìm thấy sản phẩm phù hợp cho khách hàng này"]})

        recs_list = recs.collect()[0]['recommendations']
        item_indices = [r['item_index'] for r in recs_list]

        recommended_pd = item_map.filter(F.col("item_index").isin(item_indices)).toPandas()

        final = product_info[product_info['product_id'].isin(recommended_pd['product_id'])]

        if final.empty:
            return pd.DataFrame({"Thông báo": ["Không tìm thấy chi tiết sản phẩm"]})

        return final.head(10)

    except Exception as e:
        return pd.DataFrame({"Lỗi": [str(e)]})


# ====================== KHỞI TẠO MAPPING KHI CHẠY ======================
reload_als_for_recommend()
# ========================== FP-GROWTH ==========================
fp_model = FPGrowthModel.load("/content/drive/MyDrive/BigData_Nhom5/models/fp.model_correct_v2")
rules_pd = fp_model.associationRules.toPandas()

product_map = spark.read.csv("/content/drive/MyDrive/BigData_Nhom5/data/datasetCuoiKy/olist_products_dataset.csv", header=True, inferSchema=True)\
    .select("product_id", "product_category_name")\
    .join(spark.read.csv("/content/drive/MyDrive/BigData_Nhom5/data/datasetCuoiKy/product_category_name_translation.csv", header=True, inferSchema=True),
          "product_category_name", "left")\
    .select("product_id", "product_category_name_english").toPandas()
product_map = product_map.rename(columns={"product_category_name_english": "product_name"}).fillna("Unknown")

def get_association_rules(product_input: str, top_n: int = 5):
    if not product_input or not product_input.strip():
        return pd.DataFrame(columns=["Sản phẩm đã mua", "Sản phẩm thường mua cùng", "Confidence", "Lift"])
    product_input = product_input.strip()
    def contains_product(item_list):
        return any(product_input.lower() in str(item).lower() for item in item_list)

    filtered = rules_pd[rules_pd['antecedent'].apply(contains_product) | rules_pd['consequent'].apply(contains_product)].copy()
    if len(filtered) == 0:
        return pd.DataFrame({"Thông báo": [f"Không tìm thấy luật cho {product_input}"]})

    filtered = filtered.sort_values(by='lift', ascending=False).head(top_n)

    def map_names(items):
        names = [product_map[product_map['product_id'] == pid]['product_name'].values[0]
                 if not product_map[product_map['product_id'] == pid].empty else str(pid)[:8]+"..." for pid in items]
        return " + ".join(names)

    filtered['Sản phẩm đã mua'] = filtered['antecedent'].apply(map_names)
    filtered['Sản phẩm thường mua cùng'] = filtered['consequent'].apply(map_names)
    filtered['Confidence'] = filtered['confidence'].round(4)
    filtered['Lift'] = filtered['lift'].round(2)
    return filtered[['Sản phẩm đã mua', 'Sản phẩm thường mua cùng', 'Confidence', 'Lift']]

# ========================== ADMIN FUNCTIONS ==========================
def on_upload_clicked(file_obj):
    if file_obj is None:
        return "❌ Vui lòng chọn file trước!"
    try:
        file_name = os.path.basename(file_obj.name)
        save_path = os.path.join(DATA_PATH, file_name)
        os.makedirs(DATA_PATH, exist_ok=True)
        shutil.copy(file_obj.name, save_path)
        return f"✅ Upload thành công!\nFile: {file_name}\nLưu tại: {save_path}"
    except Exception as e:
        return f"❌ Lỗi upload: {str(e)}"

# ========================== LOAD ALS MODEL BAN ĐẦU ==========================
als = None  # Khai báo biến toàn cục trước

try:
    als = ALSModel.load(os.path.join(PATH_MODELS, "als_model_new"))
    print("✓ Đã load ALS model cũ thành công")
except Exception as e:
    print(f"⚠️ Không load được model ALS cũ: {e}")
    als = None

# ========================== HÀM RETRAIN ALS THẬT ==========================
def train_model_logic():
    global als

    try:
        data_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]
        if not data_files:
            return "❌ Không tìm thấy file CSV nào!"

        # Ưu tiên tìm file có product_id
        order_items_file = next((f for f in data_files if 'order_items' in f.lower()), None)

        if order_items_file:
            print(f"✓ Ưu tiên dùng file có sản phẩm: {order_items_file}")
            file_path = os.path.join(DATA_PATH, order_items_file)
            df = spark.read.csv(file_path, header=True, inferSchema=True)
        else:
            print("⚠️ Không tìm thấy file order_items, dùng tất cả file có sẵn")
            dfs = [spark.read.csv(os.path.join(DATA_PATH, f), header=True, inferSchema=True)
                   for f in data_files]
            df = reduce(lambda x, y: x.unionByName(y, allowMissingColumns=True), dfs)

        # Lọc dữ liệu hợp lệ
        if 'price' in df.columns:
            rating_col = 'price'
        elif 'payment_value' in df.columns:
            rating_col = 'payment_value'
        else:
            rating_col = df.columns[-1]   # cột cuối cùng

        df = df.filter(F.col(rating_col).isNotNull() & (F.col(rating_col) > 0))

        # Chọn user và item
        if 'customer_unique_id' in df.columns and 'product_id' in df.columns:
            user_col = 'customer_unique_id'
            item_col = 'product_id'
        else:
            user_col = 'order_id'
            item_col = 'product_id' if 'product_id' in df.columns else df.columns[1]

        total_rows = df.count()
        print(f"Tổng số dòng hợp lệ: {total_rows:,}")

        # Indexer
        user_indexer = StringIndexer(inputCol=user_col, outputCol="user_index", handleInvalid="keep")
        item_indexer = StringIndexer(inputCol=item_col, outputCol="item_index", handleInvalid="keep")

        indexed_df = Pipeline(stages=[user_indexer, item_indexer]).fit(df).transform(df)

        # Train ALS
        als_new = ALS(
            userCol="user_index",
            itemCol="item_index",
            ratingCol=rating_col,
            rank=40,
            maxIter=10,
            regParam=0.1,
            coldStartStrategy="drop"
        )

        model = als_new.fit(indexed_df)
        model.write().overwrite().save(ALS_MODEL_PATH)

        global als
        als = model

        return f"""✅ TRAIN ALS THÀNH CÔNG (CÓ SẢN PHẨM)

- Số dòng huấn luyện: {total_rows:,}
- User column : {user_col}
- Item column : {item_col}
- Rating column: {rating_col}
- Model rank: 40
- Model đã lưu vào Drive"""

    except Exception as e:
        return f"❌ Lỗi train ALS: {str(e)}"

# ========================== GIAO DIỆN CHÍNH ==========================
with gr.Blocks(title="Hệ Thống Olist - Nhóm 5", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛍️ HỆ THỐNG OLIST - NHÓM 5")
    gr.Markdown("**Big Data Project** | Dashboard • Phân khúc • Gợi ý • FP-Growth • Dự đoán • Admin")

    with gr.Tabs():
        with gr.TabItem("🏠 Dashboard"):
            gr.Markdown("### Tổng quan kinh doanh")
            with gr.Row():
                gr.Number(value=len(df_dashboard), label="Tổng số đơn hàng", precision=0)
                gr.Number(value=round(df_dashboard[col_review].mean(), 2) if col_review else 0, label="Điểm Review Trung bình")
            btn_load = gr.Button("🚀 CẬP NHẬT BIỂU ĐỒ", variant="primary")
            with gr.Row():
                p1 = gr.Plot(label="Doanh thu theo tháng")
                p2 = gr.Plot(label="Phân bố đánh giá")
            p3 = gr.Plot(label="Top 10 ngành hàng")
            btn_load.click(create_dashboard_plots, outputs=[p1, p2, p3])

        with gr.TabItem("👥 Phân khúc Khách hàng (RFM)"):
            gr.Markdown("Upload file CSV chứa Recency, Frequency, Monetary")
            with gr.Row():
                file_input = gr.File(label="Upload file CSV", file_types=[".csv"])
                cluster_count = gr.Slider(2, 10, value=4, step=1, label="Số lượng nhóm (K)")
                btn_cluster = gr.Button("🚀 CHẠY PHÂN CỤM", variant="primary")
            plot_cluster = gr.Plot(label="Biểu đồ phân cụm 3D")
            with gr.Row():
                summary_table = gr.DataFrame(label="Đặc điểm từng nhóm")
                raw_table = gr.DataFrame(label="10 dòng dữ liệu")
            btn_cluster.click(handle_clustering, [file_input, cluster_count], [plot_cluster, summary_table, raw_table])

        with gr.TabItem("🛒 Gợi ý Sản phẩm (ALS)"):
            customer_id = gr.Textbox(label="Mã khách hàng (customer_unique_id)", placeholder="Ví dụ: 8d0e0879191d761df1fde63cd721065d")
            btn_rec = gr.Button("🔍 Tìm sản phẩm phù hợp", variant="primary")
            rec_output = gr.Dataframe()
            btn_rec.click(recommend_products, customer_id, rec_output)

        with gr.TabItem("📈 Xu hướng mua sắm (FP-Growth)"):
            prod_id = gr.Textbox(label="🔍 Nhập Product ID", placeholder="e53e557d5a159f5aa2c5e995dfdf244b")
            top_n = gr.Slider(1, 5, value=3, step=1, label="Số luật")
            btn_fp = gr.Button("🔥 Xem luật kết hợp", variant="primary")
            fp_table = gr.DataFrame()
            btn_fp.click(get_association_rules, [prod_id, top_n], fp_table)

        with gr.TabItem("🔮 Dự đoán"):
            gr.Markdown("### Dự báo giá trị thanh toán hoặc mức độ hài lòng")

            # Loại dự đoán
            mode = gr.Radio(
                choices=["📈 Regression (Dự báo Số tiền)", "📊 Classification (Dự báo Review)"],
                label="Loại dự đoán",
                value="📈 Regression (Dự báo Số tiền)",
                interactive=True
            )

            # ==================== NHÓM CHO REGRESSION (Dự báo Số tiền) ====================
            with gr.Group(visible=True) as group_regression:
                gr.Markdown("**🔢 Các trường cho Dự báo Số tiền**")
                with gr.Row():
                    price = gr.Number(label="Giá sản phẩm (R$)", value=100.0)
                    freight = gr.Number(label="Phí vận chuyển (R$)", value=20.0)
                    items = gr.Number(label="Số lượng", value=1, precision=0)

                with gr.Row():
                    weight = gr.Number(label="Trọng lượng (g)", value=500.0)
                    installments = gr.Slider(1, 24, value=1, step=1, label="Số lần trả góp")

            # ==================== NHÓM CHO CLASSIFICATION (Dự báo Review) ====================
            with gr.Group(visible=False) as group_classification:
                gr.Markdown("**⭐ Các trường cho Dự báo Review**")
                with gr.Row():
                    photos = gr.Slider(1, 10, value=2, step=1, label="Số lượng ảnh")
                    delay = gr.Number(label="Số ngày giao chậm", value=0.0)
                    is_late = gr.Radio([0, 1], label="Giao trễ?", value=0)

                with gr.Row():
                    pay_type = gr.Dropdown(
                        ["credit_card", "boleto", "voucher", "debit_card"],
                        label="Thanh toán", value="credit_card"
                    )
                    cat = gr.Dropdown(
                        ["bed_bath_table", "health_beauty", "sports_leisure", "computers_accessories", "furniture_decor"],
                        label="Ngành hàng", value="health_beauty"
                    )

                with gr.Row():
                    c_state = gr.Dropdown(["SP", "RJ", "MG", "RS", "PR"], label="Bang khách hàng", value="SP")
                    s_state = gr.Dropdown(["SP", "RJ", "MG", "RS", "PR"], label="Bang người bán", value="SP")
                    status = gr.Dropdown(["delivered", "shipped", "canceled"], label="Trạng thái đơn", value="delivered")

            # Nút dự đoán và kết quả
            btn_pred = gr.Button("🚀 THỰC HIỆN DỰ ĐOÁN", variant="primary", size="large")
            output_pred = gr.Textbox(label="Kết quả dự đoán từ AI", lines=6)

            # ==================== HÀM ẨN / HIỆN KHI ĐỔI LOẠI DỰ ĐOÁN ====================
            def toggle_inputs(selected_mode):
                if "Regression" in selected_mode:
                    return {
                        group_regression: gr.update(visible=True),
                        group_classification: gr.update(visible=False)
                    }
                else:  # Classification
                    return {
                        group_regression: gr.update(visible=False),
                        group_classification: gr.update(visible=True)
                    }

            # Sự kiện thay đổi Radio
            mode.change(
                fn=toggle_inputs,
                inputs=mode,
                outputs=[group_regression, group_classification]
            )

            # ==================== NÚT DỰ ĐOÁN ====================
            btn_pred.click(
                fn=predict_combined,
                inputs=[mode, price, freight, items, weight, installments,
                        pay_type, cat, c_state, s_state, status, delay, photos, is_late],
                outputs=output_pred
            )

                # ==================== TRANG ADMIN ====================
        with gr.TabItem("⚙️ Trang Admin"):
            gr.Markdown("# 🔧 TRANG QUẢN TRỊ HỆ THỐNG")

            with gr.Tabs():
                # Tab 1: Upload Data
                with gr.TabItem("📤 Upload Data"):
                    upload_file = gr.File(label="Chọn file CSV", file_types=[".csv"])
                    btn_upload = gr.Button("📤 Lưu vào Drive", variant="primary")
                    upload_status = gr.Textbox(label="Trạng thái", lines=3)
                    btn_upload.click(on_upload_clicked, inputs=upload_file, outputs=upload_status)

                # Tab 2: Retrain Model (ALS)
                with gr.TabItem("🔄 Retrain Model (ALS)"):
                    gr.Markdown("**Lưu ý**: Nút này sẽ dùng tất cả file CSV bạn đã upload để huấn luyện lại model ALS.")
                    btn_retrain = gr.Button("🚀 Bắt đầu huấn luyện lại Model ALS",
                                          variant="secondary", size="large")
                    retrain_status = gr.Textbox(label="Kết quả huấn luyện", lines=6)

                    # Chỉ click 1 lần duy nhất
                    btn_retrain.click(
                        fn=train_model_logic,
                        inputs=None,
                        outputs=retrain_status
                    )

                # Tab 3: Báo cáo Model
                with gr.TabItem("📊 Báo cáo Model"):
                    btn_report = gr.Button("📈 Xem báo cáo Model hiện tại", variant="primary")
                    report_output = gr.Textbox(label="Thông tin Model", lines=10)
                    btn_report.click(
                        fn=show_report,
                        inputs=None,
                        outputs=report_output
                    )

demo.launch(share=True, debug=True)

✓ Load model ALS mới từ train thành công
✓ Mapping sẵn sàng - Số user: 96,096 | Số item: 31,882
✓ Đã load ALS model cũ thành công


/tmp/ipykernel_3509/1733018051.py:399: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://da6814d81aa16e01d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_3509/1733018051.py:118: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/sta

✓ Ưu tiên dùng file có sản phẩm: olist_order_items_dataset.csv
Tổng số dòng hợp lệ: 112,650


/tmp/ipykernel_3509/1733018051.py:118: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

/tmp/ipykernel_3509/1733018051.py:118: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

/tmp/ipykernel_3509/1733018051.py:118: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.

✓ Ưu tiên dùng file có sản phẩm: olist_order_items_dataset.csv
Tổng số dòng hợp lệ: 112,650
